# Endpoint Testing: Idempotent Validation

This notebook validates the deployed serving endpoint.

## Idempotency
- ✅ **Read-only validation** - doesn't modify state
- ✅ Tests can be run multiple times
- ✅ No side effects on endpoint or data
- ✅ Safe to re-run after failures

## Test Types
- **smoke**: Quick tests (5 tests, ~2 min) - used for PROD
- **full**: Comprehensive tests (15 tests, ~10 min) - used for QA

## Test Coverage
1. Endpoint exists and is ready
2. Model serving configuration
3. Response latency
4. Response format validation
5. Error handling

In [0]:
import requests
import time
import json
from datetime import datetime

In [0]:
#  Parameters
dbutils.widgets.text("serving_endpoint_name", "workday_sales_rag_endpoint")

serving_endpoint_name = dbutils.widgets.get("serving_endpoint_name")

# Get workspace URL and token
workspace_url = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# API endpoints
management_url = f"{workspace_url}/api/2.0/serving-endpoints/{serving_endpoint_name}"  # For GET requests (status, config)
invocation_url = f"{workspace_url}/serving-endpoints/{serving_endpoint_name}/invocations"  # For POST requests (queries)

print(f"Management URL: {management_url}")
print(f"Invocation URL: {invocation_url}")

In [0]:
def wait_for_endpoint_ready(max_wait=900, poll_interval=10):
    """Wait for the serving endpoint to be in READY state before querying."""
    print(f"⏳ Waiting for endpoint '{serving_endpoint_name}' to be READY...")
    elapsed = 0
    
    while elapsed < max_wait:
        response = requests.get(management_url, headers={"Authorization": f"Bearer {token}"})
        if response.status_code == 200:
            endpoint_info = response.json()
            state = endpoint_info.get('state', {}).get('ready')
            print(f"   [{elapsed}s] State: {state}")
            
            if state == 'READY':
                print("✅ Endpoint is READY!")
                return True
        else:
            print(f"   [{elapsed}s] GET failed: {response.status_code}")
        
        time.sleep(poll_interval)
        elapsed += poll_interval
    
    print(f"❌ Endpoint not ready after {max_wait}s")
    return False

state = wait_for_endpoint_ready()

if state == False:
    raise Exception("Endpoint not ready")

In [0]:
def test_endpoint_exists():
    """Check if endpoint exists"""
    response = requests.get(management_url, headers={"Authorization": f"Bearer {token}"})
    if response.status_code == 200:
        endpoint_info = response.json()
        print(f"   Endpoint ID: {endpoint_info.get('id')}")
        print(f"   State: {endpoint_info.get('state', {}).get('config_update')}")
        return True
    return False

endpoint_exists = test_endpoint_exists()
print(endpoint_exists)

In [0]:
def test_endpoint_ready():
    """Check if endpoint is in READY state"""
    response = requests.get(management_url, headers={"Authorization": f"Bearer {token}"})
    if response.status_code == 200:
        endpoint_info = response.json()
        state = endpoint_info.get('state', {}).get('ready')
        print(f"   Ready state: {state}")
        return state == 'READY'
    return False

endpoint_ready = test_endpoint_ready()
print(endpoint_ready)

In [0]:
def test_model_config():
    """Verify model is configured correctly"""
    response = requests.get(management_url, headers={"Authorization": f"Bearer {token}"})
    if response.status_code == 200:
        endpoint_info = response.json()
        entities = endpoint_info.get('config', {}).get('served_entities', [])
        if entities:
            entity = entities[0]
            print(f"   Model: {entity.get('entity_name')}")
            print(f"   Version: {entity.get('entity_version')}")
            print(f"   Workload: {entity.get('workload_type')} / {entity.get('workload_size')}")
            return True
    return False

model_config = test_model_config()
print(model_config)

In [0]:
def query_endpoint(question):
    """Test basic endpoint query with standard OpenAI chat format"""
    # Wait for endpoint to be ready before querying
    if not wait_for_endpoint_ready():
        return False
    
    payload = {"messages": [{"role": "user", "content": question}]}
    header = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    response = requests.post(invocation_url, headers=header, json=payload, timeout=120)
    
    if response.status_code == 200:
        result = response.json()
        print(f"   Response: {result}")
        # Extract answer from OpenAI format response
        try:
            if 'choices' in result and len(result['choices']) > 0:
                answer = result['choices'][0]['message']['content']
                print(f"   Answer: {answer[:200]}...")
        except (KeyError, IndexError) as e:
            print(f"   Could not extract answer: {e}")
        print(f"   Response length: {len(str(result))} characters")
        return True
    else:
        print(f"   Status code: {response.status_code}")
        print(f"   Response: {response.text}")
        return False

query_endpoint("What are the key features of our product?")

